# Did my DiD recover the true ATE under selection bias?

This notebook uses `spuriosity` to generate panel data with a **known, true treatment effect**, then checks whether a standard difference-in-differences (DiD) estimator recovers it — first under clean conditions, then under **attrition bias**: low-outcome treated units becoming more likely to drop out of the sample after the policy takes effect.

Because we control the data-generating process, we know the *true* effect exactly (unlike with real data, where you never actually know the ground truth) — so we can directly measure how much attrition bias distorts the DiD estimate.

## Setup

In [2]:
%pip install -q "spuriosity[linearmodels] @ git+https://github.com/Nityahapani/spuriosity.git"

Note: you may need to restart the kernel to use updated packages.


In [3]:
from spuriosity import PanelGenerator, reference
import pandas as pd
import numpy as np

print("spuriosity imported successfully")

spuriosity imported successfully


## Part 1: Clean DiD, no selection bias

We construct a panel where:
- `policy_group` is fixed group membership from period 0 (who is ever treated)
- The actual **effect** only kicks in at period 5, via a structural break on `policy_group`'s coefficient

This two-step construction matters: `add_treatment`'s `start_period` controls when the *treatment column itself* switches on. If we used `start_period=5` to mean "the effect begins at period 5," the treatment column would be collinear with the DiD `post` indicator (both would be 0 before period 5 and only vary together after) — breaking the standard `treatment * post` DiD regression. Using a structural break for the effect timing avoids this.

In [5]:
gen = PanelGenerator(n_entities=8000, n_periods=10, seed=42)
gen.add_variable("income", dist="normal", mean=50000, std=15000)
gen.add_treatment("policy_group", assignment="random", start_period=0, propensity=0.5)
gen.set_outcome(
    formula="income + policy_group",
    coefficients={"income": 0.0001, "policy_group": 0.0, "Intercept": 1.0},
    noise_std=0.5,
)
gen.add_structural_break(
    period=5, target="y", kind="coefficient_shift", magnitude=3.0, coefficient_target="policy_group"
)
df_clean, truth_clean = gen.generate()
df_clean.head()

,entity_id,period,income,policy_group,y
0,0,0,56274.949480,1,5.802795
1,0,1,59083.642606,1,7.611364
2,0,2,50431.817899,1,6.404917
3,0,3,33736.310012,1,3.959098
4,0,4,71963.314649,1,8.438266


In [6]:
print(truth_clean)

GroundTruth(seed=42, spuriosity='0.1.0')
  true_coefficients: {'income': 0.0001, 'policy_group': 0.0, 'Intercept': 1.0}
  break_points: 1 (['coefficient_shift'])
  treatment_effect_ate: 0.0000
  has_true_cate: False


In [7]:
fit_clean = reference.did_fit(df_clean, outcome="y", treatment="policy_group", period="period", post_period=5)
print("DiD estimate (clean data):", round(fit_clean.ate_estimate, 3))
print("True effect: 3.0")

DiD estimate (clean data): 2.937
True effect: 3.0


The DiD estimate is close to the true effect of 3.0, as expected on clean data with no confounding or selection issues.

## Part 2: Same DGP, with attrition bias

Now we inject `SelectionBias`: treated units with a **low post-treatment outcome** are much more likely to drop out of the sample (`drop_prob=0.9`). This models a realistic scenario — e.g. people who don't benefit from a program disengage and stop being observed.

In [10]:
gen2 = PanelGenerator(n_entities=8000, n_periods=10, seed=42)
gen2.add_variable("income", dist="normal", mean=50000, std=15000)
gen2.add_treatment("policy_group", assignment="random", start_period=0, propensity=0.5)
gen2.set_outcome(
    formula="income + policy_group",
    coefficients={"income": 0.0001, "policy_group": 0.0, "Intercept": 1.0},
    noise_std=0.5,
)
gen2.add_structural_break(
    period=5, target="y", kind="coefficient_shift", magnitude=3.0, coefficient_target="policy_group"
)
gen2.add_selection_bias(rule="(policy_group == 1) and (y < 5)", drop_prob=0.9)
df_biased, truth_biased = gen2.generate()

print("Clean data rows:   ", len(df_clean))
print("Attrition-biased rows:", len(df_biased))

Clean data rows:    80000
Attrition-biased rows: 75221


In [11]:
fit_biased = reference.did_fit(df_biased, outcome="y", treatment="policy_group", period="period", post_period=5)
print("DiD estimate (attrition-biased data):", round(fit_biased.ate_estimate, 3))
print("True effect: 3.0")
print("Bias:", round(fit_biased.ate_estimate - 3.0, 3))

DiD estimate (attrition-biased data): 2.344
True effect: 3.0
Bias: -0.656


## Takeaway

In [13]:
print(f"Clean DiD estimate:     {fit_clean.ate_estimate:.3f}  (true: 3.000)")
print(f"Attrition-biased DiD:   {fit_biased.ate_estimate:.3f}  (true: 3.000)")

Clean DiD estimate:     2.937  (true: 3.000)
Attrition-biased DiD:   2.344  (true: 3.000)


Attrition of low-outcome treated units after the policy takes effect biases the DiD estimate away from the true effect, even though the underlying treatment effect never changed. A DiD analyst who doesn't check for differential attrition between groups would **underestimate** the policy's true impact here.

This is exactly the kind of failure mode `spuriosity` is built to let you stress-test *before* you trust a DiD result on real data: if your estimator recovers the truth on clean synthetic data but breaks under a realistic selection mechanism, that's a warning sign worth taking seriously.